In [1]:
import os
import pandas as pd

DATA_DIR = '.'
CLEAN_DIR = 'clean_data'
SAMPLE_SIZE = 80000  # change this to scale up/down

os.makedirs(CLEAN_DIR, exist_ok=True)
print('Pipeline configured. Sample size:', SAMPLE_SIZE)

Pipeline configured. Sample size: 80000


In [2]:
print('Loading patents (full file, 3 cols)...')
patents = pd.read_csv(
    os.path.join(DATA_DIR, 'g_patent.tsv', 'g_patent.tsv'),
    sep='\t', dtype=str,
    usecols=['patent_id', 'patent_title', 'patent_date'],
)
patents['year'] = pd.to_datetime(patents['patent_date'], errors='coerce').dt.year
patents = patents.dropna(subset=['year', 'patent_id'])
patents['year'] = patents['year'].astype(int)
print(f'Total patents: {len(patents):,}')
print(f'Year range: {patents["year"].min()} - {patents["year"].max()}  ({patents["year"].nunique()} years)')
patents.head()

Loading patents (full file, 3 cols)...
Total patents: 9,454,161
Year range: 1976 - 2025  (50 years)


,patent_id,patent_date,patent_title,year
0,10000000,2018-06-19,Coherent LADAR using intra-pixel quadrature de...,2018
1,10000001,2018-06-19,Injection molding machine and mold thickness c...,2018
2,10000002,2018-06-19,Method for manufacturing polymer film and co-e...,2018
3,10000003,2018-06-19,Method for producing a container from a thermo...,2018
4,10000004,2018-06-19,"Process of obtaining a double-oriented film, c...",2018


In [4]:
years = patents['year'].nunique()
per_year = max(200, SAMPLE_SIZE // years)
print(f'Sampling up to {per_year:,} patents per year across {years} years...')

# Use groupby().sample() — clean, no apply, no FutureWarning
year_counts = patents.groupby('year').size()
take_per_year = year_counts.clip(upper=per_year).to_dict()

frames = []
for year, n in take_per_year.items():
    grp = patents[patents['year'] == year]
    frames.append(grp.sample(n=n, random_state=42))
sampled = pd.concat(frames, ignore_index=True)

if len(sampled) > SAMPLE_SIZE:
    sampled = sampled.sample(SAMPLE_SIZE, random_state=42).reset_index(drop=True)

print(f'Sampled patents: {len(sampled):,}')
print(f'Years covered:  {sampled["year"].min()} - {sampled["year"].max()}  ({sampled["year"].nunique()} distinct)')
sampled['year'].value_counts().sort_index

Sampling up to 1,600 patents per year across 50 years...
Sampled patents: 80,000
Years covered:  1976 - 2025  (50 distinct)


<bound method Series.sort_index of year
1976    1600
1977    1600
1978    1600
1979    1600
1980    1600
1981    1600
1982    1600
1983    1600
1984    1600
1985    1600
1986    1600
1987    1600
1988    1600
1989    1600
1990    1600
1991    1600
1992    1600
1993    1600
1994    1600
1995    1600
1996    1600
1997    1600
1998    1600
1999    1600
2000    1600
2001    1600
2002    1600
2003    1600
2004    1600
2005    1600
2006    1600
2007    1600
2008    1600
2009    1600
2010    1600
2011    1600
2012    1600
2013    1600
2014    1600
2015    1600
2016    1600
2017    1600
2018    1600
2019    1600
2020    1600
2021    1600
2022    1600
2023    1600
2024    1600
2025    1600
Name: count, dtype: int64>

In [5]:
patents_clean = sampled[['patent_id', 'patent_title', 'patent_date', 'year']].copy()
patents_clean.columns = ['patent_id', 'title', 'filing_date', 'year']
patents_clean['abstract'] = ''
patents_clean = patents_clean[['patent_id', 'title', 'abstract', 'filing_date', 'year']]
patents_clean.to_csv(os.path.join(CLEAN_DIR, 'clean_patents.csv'), index=False)

valid_ids = set(patents_clean['patent_id'])
print(f'Saved clean_patents.csv: {len(patents_clean):,}')

Saved clean_patents.csv: 80,000


In [ ]:
## 2. Proportional random sample across the full file

Random rows from the 9.4M-row patent file. The year distribution comes from the source — more recent years naturally have more patents, so the time-series chart shows a real growth curve instead of a flat line.

In [ ]:
n = min(SAMPLE_SIZE, len(patents))
print(f'Proportional random sample: {n:,} rows of {len(patents):,}')

sampled = patents.sample(n=n, random_state=42).reset_index(drop=True)

print(f'Sampled patents: {len(sampled):,}')
print(f'Years covered:  {sampled["year"].min()} - {sampled["year"].max()}  ({sampled["year"].nunique()} distinct)')
print('\nNote: counts are weighted by source — recent years dominate, which is the real USPTO trend.')
sampled['year'].value_counts().sort_index().tail(10)

In [9]:
#load asignee
ass_df = load_filtered_chunked(
    os.path.join(DATA_DIR, 'g_assignee_disambiguated.tsv', 'g_assignee_disambiguated.tsv'),
    valid_ids,
    usecols=['patent_id', 'assignee_id', 'disambig_assignee_organization'],
)
ass_df = ass_df[ass_df['assignee_id'].notna() & ass_df['disambig_assignee_organization'].notna()]

comp_unique = ass_df[['assignee_id', 'disambig_assignee_organization']].drop_duplicates(subset=['assignee_id'])
comp_unique.columns = ['company_id', 'name']
comp_unique.to_csv(os.path.join(CLEAN_DIR, 'clean_companies.csv'), index=False)
print(f'Saved clean_companies.csv: {len(comp_unique):,}')
comp_unique.head()

  Scanned 8,751,310 rows, kept 71,247
Saved clean_companies.csv: 24,087


,company_id,name
0,ab103576-f045-4a48-8713-86301a0f9f4c,"Crosslink-D, Inc."
1,dfdf1f1a-8f22-41dc-a7f0-855bde20ca67,CANON KABUSHIKI KAISHA
2,3591423e-2b44-4d5b-8595-d43ebbe1cb89,Tokyo Shibaura Denki Kabushiki Kaisha
3,65ea45bc-4bc2-4ea3-acc9-a767c1cb5e2f,"American Bottlers Equipment Co., Inc."
4,ee316a2c-57c5-4dc2-b511-4b8072d6ad80,Dr. Ing. h.c. F. Porsche Aktiengesellschaft


In [10]:
#relationship tables
inv_rel = inv_df[['patent_id', 'inventor_id']].drop_duplicates().copy()
inv_rel['company_id'] = None

ass_rel = (
    ass_df[['patent_id', 'assignee_id']]
    .drop_duplicates()
    .rename(columns={'assignee_id': 'company_id'})
    .copy()
)
ass_rel['inventor_id'] = None

relationships = pd.concat([inv_rel, ass_rel], ignore_index=True)[['patent_id', 'inventor_id', 'company_id']]
relationships.to_csv(os.path.join(CLEAN_DIR, 'relationships.csv'), index=False)
print(f'Saved relationships.csv: {len(relationships):,}')

Saved relationships.csv: 254,465


In [12]:
print('Pipeline complete!')
print(f'  Patents:       {len(patents_clean):,}')
print(f'  Inventors:     {len(inv_unique):,}')
print(f'  Companies:     {len(comp_unique):,}')
print(f'  Relationships: {len(relationships):,}')


Pipeline complete!
  Patents:       80,000
  Inventors:     163,608
  Companies:     24,087
  Relationships: 254,465
